# Urbania · Fase 1: venta de departamentos en Lima

Alcance actualizado: **venta**, **departamentos**, distritos de Lima, dormitorios y precios. Verificación por navegador: 20 de octubre de 2026. No se abordan los demás filtros.

Se verificó el sitio real mediante UI y se incluye una pequeña transcripción de las cards observadas. El acceso HTTP directo anterior recibió 403; el navegador de esta sesión sí permitió navegar. Una nueva prueba HTTP del 20-10-2026 obtuvo otra vez 403, con desafío de Cloudflare. El adaptador HTTP sigue pendiente de una descarga exitosa en el entorno de despliegue. En `deliveries/week10/`, `update.py` corre a diario con GitHub Actions (`.github/workflows/urbania-refresh.yml`) y alimenta el catálogo de la web; mientras el sitio responda 403, cada ejecución se detiene sin datos nuevos. **No se certificó una captura masiva en producción.**

El notebook es autónomo. El paquete complementario contiene el módulo, cliente de captura, actualización periódica, pruebas, muestra y guía de ejecución. Las celdas de ejemplo y pruebas no hacen solicitudes de red.

## Contrato de URLs comprobado

Ruta de ejemplo: `https://urbania.pe/buscar/venta-de-departamentos-en-san-juan-de-lurigancho--lima--lima`

| Selección | Query observada |
|---|---|
| Hasta un dormitorio | `bedroomMin=0&bedroomMax=1` |
| Exactamente dos dormitorios | `bedroomsNumber=2` |
| Soles, mínimo y máximo | `priceMin=100000&priceMax=300000&currencyId=6` |
| USD, mínimo y máximo | `priceMin=50000&priceMax=100000&currencyId=2` |
| Segunda página | `page=2` |

`bedroom` significa **dormitorios**, no baños. Los parámetros se observaron al operar la UI; el generador extiende ese contrato, sin afirmar que se hayan abierto todas las combinaciones.

La paginación es importante: los enlaces observados mostraban el mismo `href` de la primera página y la navegación mediante UI añadió `?page=2`. No usar el sufijo `-pagina-2` del original ni seguir únicamente esos `href`.

Slugs observados en los enlaces del sitio: `santiago-de-surco`, `magdalena` y `ate-vitarte`. El catálogo contiene 20 distritos como cobertura inicial configurable; no pretende ser exhaustivo ni una clasificación de importancia comercial.

## Dataset limpio desde el origen

- Descartar de la base principal la sección **“Encontramos otros inmuebles que podrían interesarte”**, detectada también por `.thin-postings-container`. Contiene cards de otros distritos.
- Conservar **proyectos** por separado de avisos individuales. Un proyecto de 1 a 3 dormitorios puede incluir una tipología de 1 dormitorio; no sabemos que su precio “desde” corresponda a esa unidad. La etiqueta “En planos” no es por sí sola un criterio suficiente.
- Guardar datos publicados y procedencia. El filtro no completa los atributos ausentes: aparecer en “hasta 1” no prueba tener un dormitorio.
- Conservar nulos y rangos. No imputar mantenimiento en cero ni transformar un rango de proyecto en una unidad puntual.
- Una card sin precio, características o ubicación se conserva con nulos y bandera, sin detener la página. Si a más de la mitad de las cards principales les falta el mismo bloque, la página se rechaza como probable cambio de HTML.
- Barrer primero cada distrito sin filtros; dormitorios/precios sirven para contrastar cobertura o particionar cuando haga falta. Mantener un recorrido sin filtros para no perder anuncios con atributos no informados.

Ejemplo reproducido en UI: [hasta 1 dormitorio, S/100000–S/300000](https://urbania.pe/buscar/venta-de-departamentos-en-san-juan-de-lurigancho--lima--lima?bedroomMin=0&bedroomMax=1&priceMin=100000&priceMax=300000&currencyId=6).

## Dependencias y núcleo reutilizable

Python 3.10+. Si faltan dependencias en el kernel:

```python
%pip install beautifulsoup4 lxml requests
```

SQLite conserva altas, contenido actual, versiones y procedencia entre ejecuciones. La llave es `(source, posting_id)`, no la URL de búsqueda. Dos IDs distintos del mismo inmueble necesitan una deduplicación semántica posterior; no se fusionan aquí.

In [1]:
"""Urbania fase 1: venta/departamentos, filtros y estado incremental. Python 3.10+."""
from __future__ import annotations
import argparse
import csv
import hashlib
import json
import re
import sqlite3
import time
import unicodedata
from dataclasses import dataclass
from datetime import datetime, timezone, timedelta
from pathlib import Path
from urllib.parse import urlencode, urljoin, urlsplit, urlunsplit, parse_qsl
from bs4 import BeautifulSoup

BASE = 'https://urbania.pe'
VERSION = '1.0-phase1'
CURRENCY_IDS = {'PEN': 6, 'USD': 2}
# Estos slugs se observaron en los enlaces de distrito del sitio. SJL se abrió directamente.
DISTRICTS = {
 'miraflores':'Miraflores', 'santiago-de-surco':'Santiago de Surco',
 'san-isidro':'San Isidro', 'san-miguel':'San Miguel', 'san-borja':'San Borja',
 'surquillo':'Surquillo', 'jesus-maria':'Jesús María', 'magdalena':'Magdalena',
 'barranco':'Barranco', 'lima-cercado':'Lima Cercado', 'chorrillos':'Chorrillos',
 'pueblo-libre':'Pueblo Libre', 'lince':'Lince', 'la-victoria':'La Victoria',
 'la-molina':'La Molina', 'brena':'Breña', 'ate-vitarte':'Ate Vitarte',
 'los-olivos':'Los Olivos', 'san-martin-de-porres':'San Martín de Porres',
 'san-juan-de-lurigancho':'San Juan de Lurigancho',
}

def utcnow():
 return datetime.now(timezone.utc).isoformat()

def normalized(s):
 return re.sub(r'[^a-z0-9]+',' ',unicodedata.normalize('NFKD',s or '').encode('ascii','ignore').decode().lower()).strip()

@dataclass(frozen=True)
class Search:
 district: str
 bedrooms_min: int | None = None
 bedrooms_max: int | None = None
 bedrooms_exact: int | None = None
 price_min: int | None = None
 price_max: int | None = None
 currency: str | None = None

 def __post_init__(self):
  if self.district not in DISTRICTS: raise ValueError('Distrito fuera del catálogo de fase 1')
  if self.bedrooms_exact is not None and (self.bedrooms_min is not None or self.bedrooms_max is not None):
   raise ValueError('Usar dormitorios exactos o rango, no ambos')
  for value in [self.bedrooms_min,self.bedrooms_max,self.bedrooms_exact]:
   if value is not None and (isinstance(value,bool) or not isinstance(value,int) or not 0 <= value <= 5):
    raise ValueError('Dormitorios entre 0 y 5; 5+ requiere verificar otra semántica')
  for value in [self.price_min,self.price_max]:
   if value is not None and (isinstance(value,bool) or not isinstance(value,int) or value < 0):
    raise ValueError('Precios enteros no negativos')
  if self.bedrooms_min is not None and self.bedrooms_max is not None and self.bedrooms_min > self.bedrooms_max:
   raise ValueError('Rango de dormitorios invertido')
  if self.price_min is not None and self.price_max is not None and self.price_min > self.price_max:
   raise ValueError('Rango de precios invertido')
  if (self.price_min is not None or self.price_max is not None) and self.currency not in CURRENCY_IDS:
   raise ValueError('Especificar PEN o USD para filtrar precios')
  if self.currency is not None and self.currency not in CURRENCY_IDS: raise ValueError('Moneda no soportada')

 @property
 def unfiltered(self):
  return all(v is None for v in [self.bedrooms_min,self.bedrooms_max,self.bedrooms_exact,self.price_min,self.price_max,self.currency])

 def url(self, page=1):
  if isinstance(page,bool) or not isinstance(page,int) or page < 1: raise ValueError('Página >= 1')
  query={}
  if self.bedrooms_exact is not None: query['bedroomsNumber']=self.bedrooms_exact
  elif self.bedrooms_min is not None or self.bedrooms_max is not None:
   query['bedroomMin']=self.bedrooms_min if self.bedrooms_min is not None else 0
   if self.bedrooms_max is not None: query['bedroomMax']=self.bedrooms_max
  for key,value in [('priceMin',self.price_min),('priceMax',self.price_max)]:
   if value is not None: query[key]=value
  if self.currency: query['currencyId']=CURRENCY_IDS[self.currency]
  if page > 1: query['page']=page
  path=f'{BASE}/buscar/venta-de-departamentos-en-{self.district}--lima--lima'
  return path + ('?'+urlencode(query) if query else '')

 @property
 def key(self):
  return hashlib.sha256(self.url().encode()).hexdigest()

 @classmethod
 def from_url(cls,url):
  # Solo acepta URLs que el propio generador reproduce exactamente.
  parts=urlsplit(url);m=re.fullmatch(r'/buscar/venta-de-departamentos-en-(.+)--lima--lima',parts.path)
  if not m:raise ValueError('URL de búsqueda no reconocida')
  q={k:int(v) for k,v in parse_qsl(parts.query)}
  currency={v:k for k,v in CURRENCY_IDS.items()}.get(q.get('currencyId'))
  search=cls(m[1],q.get('bedroomMin'),q.get('bedroomMax'),q.get('bedroomsNumber'),q.get('priceMin'),q.get('priceMax'),currency)
  if search.url()!=url:raise ValueError('URL con parámetros fuera del contrato verificado')
  return search


def canonical_posting(url):
 parts=urlsplit(urljoin(BASE,url or ''))
 if parts.hostname not in {'urbania.pe','www.urbania.pe'} or not parts.path.startswith('/inmueble/'):
  raise ValueError('Enlace de ficha de Urbania requerido')
 if not re.search(r'-\d+$',parts.path.rstrip('/')): raise ValueError('Ficha sin ID reconocible')
 return urlunsplit(('https','urbania.pe',parts.path.rstrip('/'),'',''))

NUMBER=r'\d+(?:[.,]\d+)*'
def number(s):
 if s is None:return None
 s=re.sub(r'\s+','',str(s))
 if not re.fullmatch(NUMBER,s):return None
 if '.' in s and ',' in s:
  dec='.' if s.rfind('.')>s.rfind(',') else ','
  s=s.replace(',' if dec=='.' else '.','').replace(dec,'.')
 elif '.' in s or ',' in s:
  sep='.' if '.' in s else ','; chunks=s.split(sep)
  if len(chunks)>2:
   if not all(len(c)==3 for c in chunks[1:]):return None
   s=''.join(chunks)
  elif len(chunks[-1])==3:s=''.join(chunks)
  else:s=s.replace(sep,'.')
 try:return float(s)
 except ValueError:return None

def bounds(text,unit):
 match=re.search(rf'({NUMBER})(?:\s*(?:a|hasta|[-–])\s*({NUMBER}))?\s*{unit}',text,re.I)
 if not match:return None,None,None
 lo=number(match[1]);hi=number(match[2]) if match[2] else lo
 return (lo if lo==hi else None),lo,hi

def text(node):return re.sub(r'\s+',' ',node.get_text(' ',strip=True)).strip() if node else None

class ParseError(RuntimeError):pass
class CaptureStopped(RuntimeError):pass

# Si un bloque falta en una card, queda nulo con bandera. Si falta en la mayoría de la
# página, lo probable es un cambio de HTML: se rechaza para no grabar nulos masivos.
CARD_BLOCKS={'price_raw':('POSTING_CARD_PRICE','precio_no_informado'),
 'features_raw':('POSTING_CARD_FEATURES','caracteristicas_no_informadas'),
 'location':('POSTING_CARD_LOCATION','ubicacion_no_informada')}

def parse_listing(html,search:Search):
 soup=BeautifulSoup(html,'lxml')
 title=text(soup.title) or ''
 if any(w in title.lower() for w in ['just a moment','access denied','verify you are human']):
  raise ParseError('Página de verificación, no listado')
 cards=soup.select('[data-qa="posting PROPERTY"], [data-qa="posting DEVELOPMENT"]')
 if not cards:raise ParseError('Sin cards reconocibles; no asumir oferta vacía')
 boundary=next((h for h in soup.find_all(re.compile('^h[1-6]$')) if 'otros inmuebles que podr' in normalized(text(h))),None)
 rows=[]; seen=set()
 for card in cards:
  recommended=bool(card.find_parent(class_='thin-postings-container'))
  if boundary and boundary in card.find_all_previous():recommended=True
  href=card.get('data-to-posting')
  if not href:
   a=card.select_one('a[href*="/inmueble/"]');href=a.get('href') if a else None
  try:url=canonical_posting(href)
  except (ValueError,TypeError):continue
  pid=re.search(r'-(\d+)$',url)[1]
  # Identidad canónica: ruta/ID; data-id discrepante se registra como bandera.
  flags=[]
  if card.get('data-id') and card['data-id']!=pid:flags.append('card_id_discrepante')
  block={}
  for field,(qa,flag) in CARD_BLOCKS.items():
   block[field]=text(card.select_one(f'[data-qa="{qa}"]')) or None
   if block[field] is None:flags.append(flag)
  price_raw,features,location=block['price_raw'],block['features_raw'],block['location']
  # Sin ubicación no se puede comprobar el distrito: queda desconocido, no excluido.
  in_district=normalized(DISTRICTS[search.district]) in normalized(location) if location else None
  if in_district is False:flags.append('ubicacion_fuera_distrito_consultado')
  # El distrito consultado solo se asigna cuando la ubicación de la card lo confirma.
  row={'posting_id':pid,'url':url,'operation':'venta','property_type':'departamento',
       'district':search.district if in_district else None,
       'listing_kind':'project' if '/proyecto/' in url else 'unit','price_raw':price_raw,
       'features_raw':features,'location':location,'address':text(card.select_one('[class*="location-address"]')),
       'price_is_from':bool(re.search('desde',price_raw or '',re.I)),
       'recommendation':recommended,'in_requested_district':in_district,'quality_flags':flags}
  for cur,pattern in [('pen',r'S/\.?'),('usd',r'USD|US\$')]:
   m=re.search(rf'(?:{pattern})\s*({NUMBER})',price_raw or '',re.I);row[f'price_{cur}']=number(m[1]) if m else None
  for key,unit in [('bedrooms',r'dorm\.?'),('area_m2',r'm(?:²|2)\s*tot\.?'),('bathrooms',r'baños?')]:
   val,lo,hi=bounds(features or '',unit);row[key]=val;row[key+'_min']=lo;row[key+'_max']=hi
  if row['bedrooms'] is None and row['listing_kind']=='unit':flags.append('dormitorios_no_informados')
  if re.search(r'\bha\b',features or '',re.I):flags.append('superficie_en_hectareas_revisar')
  row['filter_match']=filter_match(row,search)
  if pid not in seen:rows.append(row);seen.add(pid)
 main=[r for r in rows if not r['recommendation']]
 for field,(qa,_) in CARD_BLOCKS.items():
  missing=sum(r[field] is None for r in main)
  if len(main)>1 and missing*2>len(main):raise ParseError(f'{qa} ausente en {missing} de {len(main)} cards; posible cambio de HTML')
 return rows

def is_main(row):
 # Resultado principal: ni recomendación ni card de otro distrito. Una card sin ubicación
 # apareció entre los resultados del distrito y cuenta como principal.
 return not row.get('recommendation') and row.get('in_requested_district') is not False

def filter_match(row,search):
 if row.get('recommendation'):return 'recommendation'
 if row.get('in_requested_district') is False:return 'different_district'
 # El rango de un proyecto no prueba una combinación precio/área/dormitorio de una unidad.
 if row['listing_kind']=='project':return 'project_review'
 unknown=row.get('in_requested_district',True) is None
 n=row.get('bedrooms')
 lo=search.bedrooms_exact if search.bedrooms_exact is not None else search.bedrooms_min
 hi=search.bedrooms_exact if search.bedrooms_exact is not None else search.bedrooms_max
 if lo is not None or hi is not None:
  if n is None:unknown=True
  elif (lo is not None and n<lo) or (hi is not None and n>hi):return 'mismatch'
 if search.price_min is not None or search.price_max is not None:
  amount=row.get('price_'+search.currency.lower())
  if amount is None or row.get('price_is_from'):unknown=True
  elif (search.price_min is not None and amount<search.price_min) or (search.price_max is not None and amount>search.price_max):return 'mismatch'
 return 'unknown' if unknown else 'match'

def rows_from_sample(sample,search:Search):
 # Filas de evidence/browser_sample.json con el formato de parse_listing.
 rows=[]
 for original in sample['rows']:
  row=dict(original);row['url']=BASE+row.pop('path')
  row.update(operation='venta',property_type='departamento')
  in_district=normalized(DISTRICTS[search.district]) in normalized(row['location'])
  row.update(in_requested_district=in_district,district=search.district if in_district else None)
  row.setdefault('price_is_from',row['listing_kind']=='project')
  row['quality_flags']=['dormitorios_no_informados'] if row['listing_kind']=='unit' and row.get('bedrooms') is None else []
  row['filter_match']=filter_match(row,search)
  rows.append(row)
 return rows

# Persistir solo datos propios de la ficha, sin filtros, posición ni tracking.
CONTENT_FIELDS=['url','operation','property_type','district','listing_kind','price_raw','features_raw','location','address',
 'price_is_from','price_pen','price_usd','bedrooms','bedrooms_min','bedrooms_max','area_m2','area_m2_min','area_m2_max',
 'bathrooms','bathrooms_min','bathrooms_max','quality_flags']

def stable_payload(row):
 payload={k:row.get(k) for k in CONTENT_FIELDS}
 payload['url']=canonical_posting(row['url'])
 return payload

class Store:
 def __init__(self,path='urbania.sqlite'):
  self.db=sqlite3.connect(path,isolation_level=None)
  self.db.row_factory=sqlite3.Row
  self.db.execute('PRAGMA foreign_keys=ON')
  self.db.execute('PRAGMA journal_mode=WAL')
  self.db.execute('PRAGMA busy_timeout=5000')
  self.db.executescript('''
   CREATE TABLE IF NOT EXISTS runs(id INTEGER PRIMARY KEY, started_at TEXT NOT NULL, finished_at TEXT, state TEXT NOT NULL);
   CREATE TABLE IF NOT EXISTS listings(source TEXT NOT NULL DEFAULT 'urbania', posting_id TEXT NOT NULL,
    first_ingested_at TEXT NOT NULL,last_ingested_at TEXT NOT NULL,last_observed_at TEXT,
    payload TEXT NOT NULL,content_hash TEXT NOT NULL,state TEXT NOT NULL DEFAULT 'active',last_seen_run INTEGER,
    PRIMARY KEY(source,posting_id));
   CREATE TABLE IF NOT EXISTS versions(source TEXT NOT NULL,posting_id TEXT NOT NULL,hash TEXT NOT NULL,
    observed_at TEXT NOT NULL,payload TEXT NOT NULL,id INTEGER PRIMARY KEY,
    FOREIGN KEY(source,posting_id) REFERENCES listings(source,posting_id));
   CREATE TABLE IF NOT EXISTS sightings(run_id INTEGER NOT NULL,posting_id TEXT NOT NULL,scope TEXT NOT NULL,
    filter_match TEXT NOT NULL,PRIMARY KEY(run_id,posting_id,scope),FOREIGN KEY(run_id) REFERENCES runs(id));
   CREATE TABLE IF NOT EXISTS members(scope TEXT NOT NULL,posting_id TEXT NOT NULL,last_run INTEGER NOT NULL,
    missing_complete_runs INTEGER NOT NULL DEFAULT 0,PRIMARY KEY(scope,posting_id));
   CREATE TABLE IF NOT EXISTS scopes(run_id INTEGER NOT NULL,scope TEXT NOT NULL,complete INTEGER NOT NULL,
    reason TEXT,PRIMARY KEY(run_id,scope));
   CREATE TABLE IF NOT EXISTS checks(id INTEGER PRIMARY KEY,posting_id TEXT NOT NULL,checked_at TEXT NOT NULL,
    http_status INTEGER NOT NULL,evidence TEXT NOT NULL);
  ''')
 def close(self):self.db.close()
 def start(self,now=None):
  return self.db.execute("INSERT INTO runs(started_at,state) VALUES(?,'running')",(now or utcnow(),)).lastrowid
 def ingest(self,run_id,search:Search,rows,observed_at=None,now=None):
  now=now or utcnow();stats={'new':0,'changed':0,'unchanged':0,'excluded':0}
  run=self.db.execute('SELECT * FROM runs WHERE id=?',(run_id,)).fetchone()
  if not run or run['state']!='running':raise ValueError('Ejecución no abierta')
  self.db.execute('BEGIN IMMEDIATE')
  try:
   for row in rows:
    if not is_main(row):stats['excluded']+=1;continue
    url=canonical_posting(row['url']);pid=re.search(r'-(\d+)$',url)[1]
    if str(row['posting_id'])!=pid:raise ValueError('posting_id no corresponde a URL')
    payload=stable_payload(row);serialized=json.dumps(payload,sort_keys=True,ensure_ascii=False)
    digest=hashlib.sha256(serialized.encode()).hexdigest()
    old=self.db.execute("SELECT * FROM listings WHERE source='urbania' AND posting_id=?",(pid,)).fetchone()
    if old and old['last_observed_at'] and (observed_at is None or observed_at<old['last_observed_at']):
     # Importar una captura antigua no debe restaurar un precio anterior ni reactivar un aviso.
     stats['excluded']+=1;continue
    if old is None:
     self.db.execute('INSERT INTO listings(source,posting_id,first_ingested_at,last_ingested_at,last_observed_at,payload,content_hash,state,last_seen_run) VALUES(?,?,?,?,?,?,?, ?,?)',
       ('urbania',pid,now,now,observed_at,serialized,digest,'active',run_id))
     stats['new']+=1;changed=True
    else:
     changed=digest!=old['content_hash'];stats['changed' if changed else 'unchanged']+=1
     self.db.execute("UPDATE listings SET last_ingested_at=?,last_observed_at=COALESCE(?,last_observed_at),payload=?,content_hash=?,state='active',last_seen_run=? WHERE source='urbania' AND posting_id=?",
       (now,observed_at,serialized,digest,run_id,pid))
    if changed:self.db.execute('INSERT INTO versions(source,posting_id,hash,observed_at,payload) VALUES(?,?,?,?,?)',('urbania',pid,digest,observed_at or now,serialized))
    self.db.execute('INSERT OR REPLACE INTO sightings VALUES(?,?,?,?)',(run_id,pid,search.key,row.get('filter_match','unknown')))
    self.db.execute('INSERT INTO members VALUES(?,?,?,0) ON CONFLICT(scope,posting_id) DO UPDATE SET last_run=excluded.last_run,missing_complete_runs=0',(search.key,pid,run_id))
   self.db.execute('COMMIT')
  except Exception:self.db.execute('ROLLBACK');raise
  return stats
 def finish_scope(self,run_id,search:Search,complete=False,reason=''):
  # Una ejecución limitada, fallida o repetida no crea evidencia de ausencia.
  old=self.db.execute('SELECT 1 FROM scopes WHERE run_id=? AND scope=?',(run_id,search.key)).fetchone()
  if old:return
  self.db.execute('BEGIN IMMEDIATE')
  try:
   self.db.execute('INSERT INTO scopes VALUES(?,?,?,?)',(run_id,search.key,int(complete),reason))
   if complete:
    self.db.execute('UPDATE members SET missing_complete_runs=missing_complete_runs+1 WHERE scope=? AND last_run<>?',(search.key,run_id))
    if search.unfiltered:
     self.db.execute("UPDATE listings SET state='not_observed' WHERE source='urbania' AND state='active' AND (last_seen_run IS NULL OR last_seen_run<>?) AND posting_id IN (SELECT posting_id FROM members WHERE scope=? AND last_run<>?)",(run_id,search.key,run_id))
   self.db.execute('COMMIT')
  except Exception:self.db.execute('ROLLBACK');raise
 def finish(self,run_id,state='finished',now=None):
  self.db.execute('UPDATE runs SET finished_at=?,state=? WHERE id=?',(now or utcnow(),state,run_id))
 def verify(self,pid,http_status,evidence='',checked_at=None,min_hours=6):
  """No realiza requests. Registra el resultado de una verificación de ficha externa."""
  checked_at=checked_at or utcnow();t=datetime.fromisoformat(checked_at)
  old=self.db.execute("SELECT * FROM listings WHERE source='urbania' AND posting_id=?",(str(pid),)).fetchone()
  if old is None:raise ValueError('Aviso desconocido')
  # Un 200 genérico, un bloqueo o un redirect no prueba actividad ni retirada.
  if http_status==200 and evidence=='valid_listing':new_state='active'
  elif http_status in {404,410} or (http_status==200 and evidence=='explicit_unavailable'):
   previous=self.db.execute('SELECT * FROM checks WHERE posting_id=? ORDER BY checked_at DESC,id DESC LIMIT 1',(str(pid),)).fetchone()
   unavailable=previous and (previous['http_status'] in {404,410} or previous['evidence']=='explicit_unavailable')
   distant=unavailable and (t-datetime.fromisoformat(previous['checked_at'])).total_seconds()>=min_hours*3600
   new_state='withdrawn' if distant else ('withdrawn' if old['state']=='withdrawn' else 'withdrawal_pending')
  else:new_state=old['state']
  self.db.execute('BEGIN IMMEDIATE')
  try:
   self.db.execute('INSERT INTO checks(posting_id,checked_at,http_status,evidence) VALUES(?,?,?,?)',(str(pid),checked_at,http_status,evidence))
   self.db.execute("UPDATE listings SET state=? WHERE source='urbania' AND posting_id=?",(new_state,str(pid)))
   self.db.execute('COMMIT')
  except Exception:self.db.execute('ROLLBACK');raise
  return new_state
 def export(self,path):
  records=[]
  for entry in self.db.execute('SELECT * FROM listings ORDER BY posting_id'):
   row=dict(entry);payload=json.loads(row.pop('payload'));row.update(payload)
   row['quality_flags']=' | '.join(row.get('quality_flags') or []);records.append(row)
  if records:
   with open(path,'w',newline='',encoding='utf-8-sig') as f:
    writer=csv.DictWriter(f,fieldnames=list(records[0]));writer.writeheader();writer.writerows(records)
  return len(records)

In [2]:
queries = {
    'Distrito, sin filtros': Search('san-juan-de-lurigancho'),
    'Hasta 1 dormitorio': Search('san-juan-de-lurigancho', bedrooms_min=0, bedrooms_max=1),
    'Exactamente 2 dormitorios': Search('san-juan-de-lurigancho', bedrooms_exact=2),
    'Dormitorios + PEN': Search('san-juan-de-lurigancho', bedrooms_min=0, bedrooms_max=1, price_min=100000, price_max=300000, currency='PEN'),
    'Dormitorios + USD': Search('san-juan-de-lurigancho', bedrooms_min=0, bedrooms_max=1, price_min=50000, price_max=100000, currency='USD'),
}
for name, query in queries.items():
    print(name, '\n', query.url())
print('Segunda página:', queries['Distrito, sin filtros'].url(2))
print('Distritos configurables:', len(DISTRICTS))

Distrito, sin filtros 
 https://urbania.pe/buscar/venta-de-departamentos-en-san-juan-de-lurigancho--lima--lima
Hasta 1 dormitorio 
 https://urbania.pe/buscar/venta-de-departamentos-en-san-juan-de-lurigancho--lima--lima?bedroomMin=0&bedroomMax=1
Exactamente 2 dormitorios 
 https://urbania.pe/buscar/venta-de-departamentos-en-san-juan-de-lurigancho--lima--lima?bedroomsNumber=2
Dormitorios + PEN 
 https://urbania.pe/buscar/venta-de-departamentos-en-san-juan-de-lurigancho--lima--lima?bedroomMin=0&bedroomMax=1&priceMin=100000&priceMax=300000&currencyId=6
Dormitorios + USD 
 https://urbania.pe/buscar/venta-de-departamentos-en-san-juan-de-lurigancho--lima--lima?bedroomMin=0&bedroomMax=1&priceMin=50000&priceMax=100000&currencyId=2
Segunda página: https://urbania.pe/buscar/venta-de-departamentos-en-san-juan-de-lurigancho--lima--lima?page=2
Distritos configurables: 20


## Muestra real observada, de cobertura parcial

Los siguientes datos se transcribieron del DOM público, sin pretender reproducir el HTML completo. Son **6 resultados principales**: un proyecto y cinco avisos individuales. Se añaden dos recomendaciones observadas para probar el descarte. Dos cards individuales no informaban dormitorios en el encabezado. No son registros inventados ni el dataset completo de Lima.

Se conoce la fecha de observación, no un timestamp exacto por card; por eso la demostración no inventa `last_observed_at`. La ejecución de ejemplo usa una base temporal para no mezclarla con producción.

In [3]:
browser_sample = {'source': 'DOM público observado en el navegador el 20-10-2026; transcripción estructurada, no HTML original completo', 'observed_date': '2026-10-20', 'query_url': 'https://urbania.pe/buscar/venta-de-departamentos-en-san-juan-de-lurigancho--lima--lima?bedroomMin=0&bedroomMax=1&priceMin=100000&priceMax=300000&currencyId=6', 'rows': [{'posting_id': '149843673', 'path': '/inmueble/proyecto/ememvein-proyecto-gran-bosque-ciruelo-lima-san-juan-de-lurigancho-los-portales-departamentos-sa-149843673', 'listing_kind': 'project', 'price_pen': 204240, 'price_usd': None, 'price_is_from': True, 'bedrooms': None, 'bedrooms_min': 1, 'bedrooms_max': 3, 'area_m2': None, 'area_m2_min': 40, 'area_m2_max': 65, 'bathrooms': None, 'price_raw': 'Departamentos desdeS/ 204,240', 'features_raw': '235 un. | 1 a 3 dorm. | 40 a 65 m² tot.', 'location': 'San Juan de Lurigancho, Lima', 'address': 'Av. Ciruelos 880', 'recommendation': False}, {'posting_id': '150834281', 'path': '/inmueble/clasificado/veclapin-venta-de-departamento-en-las-flores-san-juan-de-lurigancho-1-dormitorio-150834281', 'listing_kind': 'unit', 'price_pen': 150750, 'price_usd': 45000, 'bedrooms': 1, 'area_m2': 37, 'bathrooms': 1, 'price_raw': 'S/ 150,750 · USD 45,000', 'features_raw': '37 m² tot. | 1 dorm. | 1 baño', 'location': 'Las Flores, San Juan de Lurigancho', 'address': 'Las Alhucemas', 'recommendation': False}, {'posting_id': '148064958', 'path': '/inmueble/clasificado/veclapin-venta-de-departamento-en-san-juan-de-lurigancho-lima-1-dormitorio-148064958', 'listing_kind': 'unit', 'price_pen': 236240, 'price_usd': 68000, 'bedrooms': 1, 'area_m2': 120, 'bathrooms': 1, 'price_raw': 'S/ 236,240 · USD 68,000', 'features_raw': '120 m² tot. | 1 dorm. | 1 baño', 'location': 'San Juan de Lurigancho, Lima', 'address': 'Las Flores', 'recommendation': False}, {'posting_id': '150531968', 'path': '/inmueble/clasificado/veclapin-venta-de-departamento-en-san-juan-de-lurigancho-lima-150531968', 'listing_kind': 'unit', 'price_pen': 157500, 'price_usd': 45000, 'bedrooms': None, 'area_m2': 86, 'bathrooms': 1, 'price_raw': 'S/ 157,500 · USD 45,000', 'features_raw': '86 m² tot. | 1 baño', 'location': 'San Juan de Lurigancho, Lima', 'address': 'Av. Las Lomas', 'recommendation': False}, {'posting_id': '147866791', 'path': '/inmueble/clasificado/veclapin-venta-de-departamento-en-zarate-san-juan-de-lurigancho-147866791', 'listing_kind': 'unit', 'price_pen': 134064, 'price_usd': 39900, 'bedrooms': None, 'area_m2': 56, 'bathrooms': None, 'price_raw': 'S/ 134,064 · USD 39,900', 'features_raw': '56 m² tot.', 'location': 'Zarate, San Juan de Lurigancho', 'address': 'Los Amautas al 1100', 'recommendation': False}, {'posting_id': '146876201', 'path': '/inmueble/clasificado/veclapin-venta-de-departamento-en-san-juan-de-lurigancho-lima-1-dormitorio-146876201', 'listing_kind': 'unit', 'price_pen': 122850, 'price_usd': 35000, 'bedrooms': 1, 'area_m2': 47, 'bathrooms': 1, 'price_raw': 'S/ 122,850 · USD 35,000', 'features_raw': '47 m² tot. | 1 dorm. | 1 baño', 'location': 'San Juan de Lurigancho, Lima', 'address': 'El Periodista al 100', 'recommendation': False}, {'posting_id': '149906463', 'path': '/inmueble/proyecto/ememefin-proyecto-colonial-living-lima-cercado-mariscal-oscar-r-benavides-colindante-inmobiliaria-149906463', 'listing_kind': 'project', 'price_pen': 210000, 'bedrooms': None, 'bedrooms_min': 1, 'bedrooms_max': 3, 'area_m2': None, 'area_m2_min': 42, 'area_m2_max': 89, 'price_raw': 'Departamentos desdeS/ 210,000', 'features_raw': '206 un. | 1 a 3 dorm. | 42 a 89 m² tot.', 'location': 'Mariscal Oscar R Benavides, Lima Cercado', 'address': 'Av. Oscar R. Benavides 545, Cercado de Lima', 'recommendation': True}, {'posting_id': '149433536', 'path': '/inmueble/proyecto/ememefin-proyecto-montero-rosas-1080-lima-cercado-santa-beatriz-v-v-inmobiliaria-149433536', 'listing_kind': 'project', 'price_pen': 267000, 'bedrooms': None, 'bedrooms_min': 1, 'bedrooms_max': 3, 'area_m2': None, 'area_m2_min': 40, 'area_m2_max': 69, 'price_raw': 'Departamentos desdeS/ 267,000', 'features_raw': '280 un. | 1 a 3 dorm. | 40 a 69 m² tot.', 'location': 'Santa Beatriz, Lima Cercado', 'address': 'Jr. Montero Rosas 1080', 'recommendation': True}]}

search = Search.from_url(browser_sample['query_url'])
assert search == queries['Dormitorios + PEN']
sample_rows = rows_from_sample(browser_sample, search)
for row in sample_rows:
    print(row['posting_id'], row['listing_kind'], row['location'], 'PEN:', row.get('price_pen'), 'dormitorios:', row.get('bedrooms'), 'control:', row['filter_match'])

149843673 project San Juan de Lurigancho, Lima PEN: 204240 dormitorios: None control: project_review
150834281 unit Las Flores, San Juan de Lurigancho PEN: 150750 dormitorios: 1 control: match
148064958 unit San Juan de Lurigancho, Lima PEN: 236240 dormitorios: 1 control: match
150531968 unit San Juan de Lurigancho, Lima PEN: 157500 dormitorios: None control: unknown
147866791 unit Zarate, San Juan de Lurigancho PEN: 134064 dormitorios: None control: unknown
146876201 unit San Juan de Lurigancho, Lima PEN: 122850 dormitorios: 1 control: match
149906463 project Mariscal Oscar R Benavides, Lima Cercado PEN: 210000 dormitorios: None control: recommendation
149433536 project Santa Beatriz, Lima Cercado PEN: 267000 dormitorios: None control: recommendation


In [4]:
import tempfile
with tempfile.TemporaryDirectory() as folder:
    demo = Store(str(Path(folder) / 'demo.sqlite'))
    run1 = demo.start()
    print('Primera carga:', demo.ingest(run1, search, sample_rows))
    demo.finish_scope(run1, search, complete=False, reason='Muestra parcial')
    demo.finish(run1)
    run2 = demo.start()
    print('Misma muestra otra vez:', demo.ingest(run2, search, sample_rows))
    print('Fichas:', demo.db.execute('SELECT COUNT(*) FROM listings').fetchone()[0])
    print('Versiones:', demo.db.execute('SELECT COUNT(*) FROM versions').fetchone()[0])
    demo.finish(run2)
    demo.close()

Primera carga: {'new': 6, 'changed': 0, 'unchanged': 0, 'excluded': 2}
Misma muestra otra vez: {'new': 0, 'changed': 0, 'unchanged': 6, 'excluded': 2}
Fichas: 6
Versiones: 6


## Estados y verificación de retiradas

| Evidencia | Estado/efecto |
|---|---|
| Nueva ficha principal, mismo distrito | Alta |
| ID existente y contenido idéntico | Actualiza observación, sin duplicado ni versión adicional |
| ID existente y contenido distinto | Actualiza ficha y añade versión |
| Ausente de una consulta filtrada | No se deduce retirada |
| Ausente de un barrido completo sin filtros | `not_observed` |
| Una ficha devuelve 404/410 o indisponibilidad explícita | `withdrawal_pending` |
| Segunda confirmación separada al menos 6 horas | `withdrawn` |
| Ficha válida reaparece | `active` |
| 403, 429, error o página genérica 200 | No prueba retirada ni actividad |

El verificador registra una comprobación realizada externamente; no programa ni descarga fichas. No se verificó una eliminación real: esa lógica se prueba con respuestas simuladas. Una retirada no implica que se haya vendido el departamento.

## Adaptador HTTP acotado

La siguiente celda solo define el cliente; no inicia la captura. Lee robots, mantiene al menos 10 segundos entre peticiones y se detiene ante bloqueo. Registra HTML, estado de ejecución y métricas. No declara un barrido completo cuando se alcanza el límite, se repite una página, cambia el recuento o no se reconcilian IDs únicos con el total visible.

La parte de descarga requiere validación en el entorno de despliegue. La verificación por navegador no garantiza que este cliente HTTP tenga acceso.

In [5]:
"""Captura HTTP acotada: sin evasión de controles. No activa ninguna tarea periódica."""
import argparse
import json
import re
import time
from pathlib import Path
from urllib.parse import urlsplit, parse_qs
import requests
from bs4 import BeautifulSoup


AGENT='UrbaniaResearchComparator/1.0'
class Robots:
 def __init__(self,body):
  groups=[];agents=[];rules=[]
  for raw in body.splitlines()+['User-agent: __end__']:
   line=raw.split('#',1)[0].strip()
   if ':' not in line:continue
   k,v=[x.strip() for x in line.split(':',1)];k=k.lower()
   if k=='user-agent':
    if rules:groups.append((agents,rules));agents=[];rules=[]
    agents.append(v.lower())
   elif k in {'allow','disallow'} and agents and v:rules.append((k=='allow',v))
  specific=[(a,r) for a,r in groups if any(x!='*' and x in AGENT.lower() for x in a)]
  if specific:
   longest=max(len(x) for a,_ in specific for x in a if x!='*' and x in AGENT.lower())
   selected=[r for a,r in specific if any(len(x)==longest and x in AGENT.lower() for x in a)]
  else:selected=[r for a,r in groups if '*' in a]
  self.rules=[v for r in selected for v in r]
 def allows(self,url):
  u=urlsplit(url);target=u.path+('?' + u.query if u.query else '');hits=[]
  for allow,rule in self.rules:
   pattern='^'+re.escape(rule).replace(r'\*','.*')
   if rule.endswith('$'):pattern=pattern[:-2]+'$'
   if re.search(pattern,target):hits.append((len(rule.replace('*','').rstrip('$')),allow))
  return max(hits)[1] if hits else True

class Client:
 def __init__(self,interval=10,session=None):
  self.session=session or requests.Session();self.session.headers.update({'User-Agent':AGENT})
  self.interval=max(10,interval);self.last=0;self.stopped=False;self.robots=None
 def _request(self,url):
  if self.stopped:raise CaptureStopped('Cliente detenido')
  time.sleep(max(0,self.interval-(time.monotonic()-self.last)));self.last=time.monotonic()
  r=self.session.get(url,timeout=30,allow_redirects=False)
  if r.status_code in {401,403,429}:self.stopped=True;raise CaptureStopped(f'HTTP {r.status_code}; no reintentar ni marcar avisos retirados')
  if r.status_code!=200:raise CaptureStopped(f'HTTP {r.status_code}; cobertura incompleta')
  return r
 def fetch(self,url):
  parts=urlsplit(url)
  if parts.hostname!='urbania.pe' or not parts.path.startswith('/buscar/venta-de-departamentos'):
   raise ValueError('Solo listados de venta de departamentos')
  if self.robots is None:
   response=self._request(BASE+'/robots.txt');self.robots=Robots(response.text)
  if not self.robots.allows(url):raise CaptureStopped('Ruta excluida por robots.txt; cobertura incompleta')
  r=self._request(url)
  if 'html' not in r.headers.get('content-type','').lower():raise ParseError('Respuesta no HTML')
  if any(x in r.text[:200000].lower() for x in ['cf-chl-','just a moment...','verify you are human','error code: 1010']):
   self.stopped=True;raise CaptureStopped('Respuesta de verificación')
  return r.text,utcnow()

def continuation(html,search,page,main_count):
 soup=BeautifulSoup(html,'lxml');higher=[]
 for a in soup.select('a[href]'):
  label=a.get_text(strip=True)
  if label.isdigit() and urlsplit(a['href']).path==urlsplit(search.url()).path:
   if int(label)>page:higher.append(int(label))
 if higher:return min(higher),False
 # Ausencia de paginación no basta para asumir que un listado quedó completo.
 h=soup.find('h1');m=re.match(r'\s*([\d.,]+)\s+Departamentos',h.get_text(' ',strip=True) if h else '',re.I)
 expected=int(re.sub(r'\D','',m[1])) if m else None
 # Para una única página se verifica el recuento. En páginas finales >1, el
 # recuento global se valida al terminar el barrido mediante IDs únicos.
 return None,(expected is not None)

def capture(search,db_path='urbania.sqlite',max_pages=5,client=None,evidence_dir='captures'):
 client=client or Client();db=Store(db_path);run=db.start();events=[];fingerprints=set();ids=set();page=1;complete=False;reason='Límite de páginas';expected=None;error=None
 totals={'new':0,'changed':0,'unchanged':0,'excluded':0}
 folder=Path(evidence_dir)/str(run);folder.mkdir(parents=True,exist_ok=True)
 try:
  for index in range(max_pages):
   url=search.url(page);html,observed=client.fetch(url)
   (folder/f'page_{page}.html').write_text(html,encoding='utf-8')
   rows=parse_listing(html,search);main=[r for r in rows if is_main(r)]
   fp=tuple(sorted(r['posting_id'] for r in main))
   if fp in fingerprints:reason='Página repetida';break
   fingerprints.add(fp);ids.update(fp)
   h=BeautifulSoup(html,'lxml').find('h1');m=re.match(r'\s*([\d.,]+)\s+Departamentos',h.get_text(' ',strip=True) if h else '',re.I)
   current_expected=int(re.sub(r'\D','',m[1])) if m else None
   if expected is None:expected=current_expected
   elif current_expected!=expected:reason='Recuento cambió durante barrido';expected=-1
   stats=db.ingest(run,search,rows,observed_at=observed)
   for key in totals:totals[key]+=stats[key]
   events.append({'url':url,'observed_at':observed,'main_cards':len(main),'stats':stats})
   next_page,has_count=continuation(html,search,page,len(main))
   if next_page is None:
    complete=has_count and expected is not None and expected>=0 and len(ids)==expected
    reason='Recuento y final de paginación verificados' if complete else 'Cobertura sin reconciliar con recuento'
    break
   page=next_page
 except (requests.RequestException,CaptureStopped,ParseError) as exc:
  reason=str(exc);complete=False;error=type(exc).__name__
 finally:
  db.finish_scope(run,search,complete,reason);db.finish(run,'finished' if complete else 'partial');db.close()
  (folder/'run.json').write_text(json.dumps({'run_id':run,'search_url':search.url(),'complete':complete,'reason':reason,'error':error,'stats':totals,'events':events},ensure_ascii=False,indent=2))
 return {'run_id':run,'complete':complete,'reason':reason,'error':error,'unique_ids':len(ids),'pages':len(events),'stats':totals}

## Pruebas

Respuestas HTTP, HTML y publicaciones usados a continuación son sintéticos. Se comprueban filtros, cards incompletas, persistencia, cambios de precio, repeticiones, transacciones, capturas antiguas, 403/429, retirada y reactivación. También se comprueban el flujo de dos páginas, el límite de cobertura y el límite de paginación de robots.

In [6]:
import unittest, tempfile, json
from pathlib import Path
from unittest.mock import Mock,patch



S=Search('san-juan-de-lurigancho')
def row(pid='123',price=200000):
 return dict(posting_id=pid,url=BASE+'/inmueble/clasificado/fixture-'+pid,operation='venta',property_type='departamento',listing_kind='unit',price_pen=price,bedrooms=2,area_m2=60,price_raw='S/ '+str(price),features_raw='60 m² tot. 2 dorm.',location='San Juan de Lurigancho, Lima',in_requested_district=True,recommendation=False,filter_match='match',quality_flags=[])
def card(pid,price='S/ 200,000',features='60 m² tot. 2 dorm.',location='San Juan de Lurigancho, Lima'):
 # None omite el bloque, como una card que no lo publica.
 blocks=[('h2','POSTING_CARD_PRICE',price),('h3','POSTING_CARD_FEATURES',features),('h4','POSTING_CARD_LOCATION',location)]
 return f'<div data-qa="posting PROPERTY" data-to-posting="{row(pid)["url"]}" data-id="{pid}">'+''.join(f'<{t} data-qa="{qa}">{v}</{t}>' for t,qa,v in blocks if v is not None)+'</div>'

class Filters(unittest.TestCase):
 def test_verified_range(self):
  self.assertEqual(Search(S.district,bedrooms_min=0,bedrooms_max=1,price_min=100000,price_max=300000,currency='PEN').url(),BASE+'/buscar/venta-de-departamentos-en-san-juan-de-lurigancho--lima--lima?bedroomMin=0&bedroomMax=1&priceMin=100000&priceMax=300000&currencyId=6')
 def test_verified_usd(self):self.assertIn('currencyId=2',Search(S.district,price_min=50000,price_max=100000,currency='USD').url())
 def test_verified_exact(self):self.assertTrue(Search(S.district,bedrooms_exact=2).url().endswith('?bedroomsNumber=2'))
 def test_pagination(self):self.assertTrue(S.url(2).endswith('?page=2'))
 def test_search_from_url(self):
  q=Search(S.district,bedrooms_min=0,bedrooms_max=1,price_min=100000,price_max=300000,currency='PEN');self.assertEqual(Search.from_url(q.url()),q)
  for url in [S.url(2),BASE+'/buscar/venta-de-casas-en-lima',S.url()+'?sort=low_price']:
   with self.assertRaises(ValueError):Search.from_url(url)
 def test_district_confirmed_by_location(self):
  rows=parse_listing(card('1')+card('2',location=None),S)
  self.assertEqual(rows[0]['district'],S.district);self.assertIsNone(rows[1]['district']);self.assertEqual(stable_payload(rows[0])['district'],S.district)
 def test_invalid(self):
  for kwargs in [dict(bedrooms_min=3,bedrooms_max=1),dict(price_min=10),dict(bedrooms_exact=2,bedrooms_min=1),dict(price_min=100,price_max=10,currency='PEN')]:
   with self.assertRaises(ValueError):Search(S.district,**kwargs)
 def test_url_tracking_removed(self):self.assertEqual(canonical_posting(row()['url']+'?n_src=Listado'),row()['url'])
 def test_numbers(self):
  for raw,value in [('114.17',114.17),('2,500.50',2500.5),('2.500,50',2500.5),('1,200',1200)]:self.assertEqual(number(raw),value)
 def test_unknown_not_one(self):self.assertEqual(bounds('86 m² tot. 1 baño',r'dorm\.?'),(None,None,None))
 def test_range_preserved(self):self.assertEqual(bounds('1 a 3 dorm.',r'dorm\.?'),(None,1,3))
 def test_project_review(self):r=row();r['listing_kind']='project';self.assertEqual(filter_match(r,Search(S.district,bedrooms_exact=1)),'project_review')
 def test_mismatch(self):self.assertEqual(filter_match(row(),Search(S.district,bedrooms_exact=1)),'mismatch')
 def test_unknown_currency(self):r=row();self.assertEqual(filter_match(r,Search(S.district,price_max=100000,currency='USD')),'unknown')
 def test_recommendations(self):
  rows=parse_listing(card('1')+'<h2>Encontramos otros inmuebles que podrían interesarte</h2><div class="thin-postings-container">'+card('2')+'</div>',S)
  self.assertFalse(rows[0]['recommendation']);self.assertTrue(rows[1]['recommendation'])
 def test_incomplete_card_kept(self):
  rows=parse_listing(card('1')+card('2',features=None)+card('3',price=None),S)
  self.assertEqual(len(rows),3);self.assertIsNone(rows[1]['bedrooms']);self.assertIn('caracteristicas_no_informadas',rows[1]['quality_flags']);self.assertEqual(rows[1]['price_pen'],200000)
  self.assertIsNone(rows[2]['price_pen']);self.assertFalse(rows[2]['price_is_from']);self.assertIn('precio_no_informado',rows[2]['quality_flags'])
 def test_missing_location_unknown_not_excluded(self):
  r=parse_listing(card('1')+card('2',location=None),S)[1]
  self.assertIsNone(r['in_requested_district']);self.assertIn('ubicacion_no_informada',r['quality_flags']);self.assertTrue(is_main(r));self.assertEqual(r['filter_match'],'unknown')
 def test_majority_missing_is_html_change(self):
  with self.assertRaises(ParseError):parse_listing(card('1',price=None)+card('2',price=None)+card('3'),S)
 def test_single_incomplete_card_page(self):self.assertEqual(len(parse_listing(card('1',features=None),S)),1)
 def test_robots(self):
  r=Robots('User-agent: *\nDisallow: /avisos-api/\nAllow: *page=2\nDisallow: *page=*')
  self.assertTrue(r.allows(S.url(2)));self.assertFalse(r.allows(S.url(6)))
 def test_robots_observed_page_limit(self):
  # Reglas de paginación de robots.txt observadas el 20-10-2026.
  r=Robots('User-agent: *\n'+''.join(f'Allow: *page={n}\n' for n in range(2,6))+'Disallow: *page=*')
  self.assertTrue(r.allows(S.url()));self.assertTrue(r.allows(S.url(5)));self.assertFalse(r.allows(S.url(6)))
 def test_http403_stops(self):
  session=Mock();session.headers={};session.get.return_value=Mock(status_code=403)
  c=Client(session=session)
  with patch('time.sleep'):
   for _ in range(2):
    with self.assertRaises(CaptureStopped):c.fetch(S.url())
  self.assertEqual(session.get.call_count,1)

class Persistence(unittest.TestCase):
 def setUp(self):self.tmp=tempfile.TemporaryDirectory();self.db=Store(str(Path(self.tmp.name)/'test.sqlite'))
 def tearDown(self):self.db.close();self.tmp.cleanup()
 def add(self,r=None,scope=S,observed='2026-10-20T10:00:00+00:00'):
  run=self.db.start();stats=self.db.ingest(run,scope,[r or row()],observed);return run,stats
 def count(self,table):return self.db.db.execute('SELECT COUNT(*) FROM '+table).fetchone()[0]
 def test_idempotent(self):
  run,_=self.add();stats=self.db.ingest(run,S,[row(),row()]);self.assertEqual(self.count('listings'),1);self.assertEqual(self.count('versions'),1);self.assertEqual(self.count('sightings'),1)
 def test_changed_price_versions(self):
  self.add();_,stats=self.add(row(price=210000));self.assertEqual(stats['changed'],1);self.assertEqual(self.count('versions'),2)
 def test_change_then_revert_preserved(self):
  self.add();self.add(row(price=210000));self.add();self.assertEqual(self.count('versions'),3)
 def test_cross_query_no_duplicate(self):
  self.add();self.add(scope=Search(S.district,bedrooms_exact=2));self.assertEqual(self.count('listings'),1);self.assertEqual(self.count('versions'),1)
 def test_recommendation_excluded(self):
  r=row();r['recommendation']=True;self.add(r);self.assertEqual(self.count('listings'),0)
 def test_card_without_location_ingested(self):
  r=row();r.update(location=None,in_requested_district=None);self.add(r);self.assertEqual(self.count('listings'),1)
 def test_partial_not_absent(self):
  self.add();run=self.db.start();self.db.finish_scope(run,S,False,'403');self.assertEqual(self.db.db.execute('SELECT state FROM listings').fetchone()[0],'active')
 def test_filtered_absence_not_withdrawn(self):
  filtered=Search(S.district,price_max=250000,currency='PEN');self.add(scope=filtered);run=self.db.start();self.db.finish_scope(run,filtered,True);self.assertEqual(self.db.db.execute('SELECT state FROM listings').fetchone()[0],'active')
 def test_full_absence_only_not_observed(self):
  self.add();run=self.db.start();self.db.finish_scope(run,S,True);self.assertEqual(self.db.db.execute('SELECT state FROM listings').fetchone()[0],'not_observed')
 def test_404_confirmations_and_reactivation(self):
  self.add();self.assertEqual(self.db.verify('123',404,checked_at='2026-10-20T10:00:00+00:00'),'withdrawal_pending')
  self.assertEqual(self.db.verify('123',404,checked_at='2026-10-20T10:01:00+00:00'),'withdrawal_pending')
  self.assertEqual(self.db.verify('123',410,checked_at='2026-10-21T10:00:00+00:00'),'withdrawn')
  self.assertEqual(self.db.verify('123',200,'valid_listing',checked_at='2026-10-21T11:00:00+00:00'),'active')
 def test_block_not_withdrawn(self):
  self.add()
  for status in [403,429,500,302]:self.assertEqual(self.db.verify('123',status),'active')
 def test_invalid200_not_reactivate(self):
  self.add();self.db.db.execute("UPDATE listings SET state='withdrawn'");self.assertEqual(self.db.verify('123',200,'generic_page'),'withdrawn')
 def test_stale_capture_does_not_restore(self):
  self.add(row(price=210000),observed='2026-10-21T10:00:00+00:00');self.add(row(price=200000));self.assertEqual(json.loads(self.db.db.execute('SELECT payload FROM listings').fetchone()[0])['price_pen'],210000)
 def test_transaction_rollback(self):
  run=self.db.start();bad=row('124');bad['posting_id']='wrong'
  with self.assertRaises(ValueError):self.db.ingest(run,S,[row(),bad])
  self.assertEqual(self.count('listings'),0)
 def test_closed_run_rejected(self):
  run,_=self.add();self.db.finish(run)
  with self.assertRaises(ValueError):self.db.ingest(run,S,[row()])
 def test_export(self):
  self.add();p=Path(self.tmp.name)/'out.csv';self.assertEqual(self.db.export(p),1);self.assertIn('price_pen',p.read_text(encoding='utf-8-sig'))

class CaptureIntegration(unittest.TestCase):
 def fixture(self,pid,total=2,next_page=False):
  r=row(pid)
  return f'<h1>{total} Departamentos en venta en San Juan de Lurigancho</h1><div data-qa="posting PROPERTY" data-id="{pid}" data-to-posting="{r["url"]}"><h2 data-qa="POSTING_CARD_PRICE">S/ 200,000</h2><h3 data-qa="POSTING_CARD_FEATURES">60 m² tot. 2 dorm.</h3><h4 data-qa="POSTING_CARD_LOCATION">San Juan de Lurigancho, Lima</h4></div>'+('<a href="'+S.url()+'">2</a>' if next_page else '')
 def test_two_pages_complete(self):
  fake=Mock();fake.fetch.side_effect=[(self.fixture('1',next_page=True),'2026-10-20T10:00:00+00:00'),(self.fixture('2'),'2026-10-20T10:01:00+00:00')]
  with tempfile.TemporaryDirectory() as folder:
   result=capture(S,str(Path(folder)/'db'),3,fake,str(Path(folder)/'html'))
   self.assertTrue(result['complete']);self.assertEqual(result['unique_ids'],2);self.assertEqual(result['pages'],2)
 def test_limit_incomplete(self):
  fake=Mock();fake.fetch.return_value=(self.fixture('1',next_page=True),'2026-10-20T10:00:00+00:00')
  with tempfile.TemporaryDirectory() as folder:
   result=capture(S,str(Path(folder)/'db'),1,fake,str(Path(folder)/'html'))
   self.assertFalse(result['complete'])
 def test_block_does_not_mark_empty_complete(self):
  fake=Mock();fake.fetch.side_effect=CaptureStopped('403')
  with tempfile.TemporaryDirectory() as folder:
   result=capture(S,str(Path(folder)/'db'),3,fake,str(Path(folder)/'html'))
   self.assertFalse(result['complete']);self.assertEqual(result['unique_ids'],0)
 def test_repeated_pages_incomplete(self):
  fake=Mock();fake.fetch.return_value=(self.fixture('1',next_page=True),'2026-10-20T10:00:00+00:00')
  with tempfile.TemporaryDirectory() as folder:
   result=capture(S,str(Path(folder)/'db'),3,fake,str(Path(folder)/'html'))
   self.assertFalse(result['complete']);self.assertEqual(result['unique_ids'],1)
 def test_incomplete_card_does_not_stop(self):
  fake=Mock();fake.fetch.return_value=('<h1>2 Departamentos en venta en San Juan de Lurigancho</h1>'+card('1')+card('2',features=None),'2026-10-20T10:00:00+00:00')
  with tempfile.TemporaryDirectory() as folder:
   result=capture(S,str(Path(folder)/'db'),3,fake,str(Path(folder)/'html'))
   self.assertTrue(result['complete']);self.assertEqual(result['unique_ids'],2);self.assertIsNone(result['error'])

suite = unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromTestCase(cls) for cls in [Filters, Persistence, CaptureIntegration])
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful()

...........................................
----------------------------------------------------------------------
Ran 43 tests in 0.501s

OK


## Siguiente fase y uso periódico

Para capturar realmente, usar `capture.py` del paquete o llamar explícitamente a `capture(Search(...), db_path='urbania.sqlite')`. El valor predeterminado limita a cinco páginas por consulta, lo mismo que permite `robots.txt`; cualquier límite o restricción deja cobertura parcial y no marca publicaciones como retiradas. El código no se ejecuta automáticamente al abrir el notebook.

Para la ejecución periódica, el paquete incluye `update.py`: recorre sin filtros todo el catálogo con un único cliente, usa un candado para que no haya dos procesos sobre la misma base, se detiene ante un bloqueo, guarda un resumen y regenera el CSV. En `deliveries/week10/` lo ejecuta a diario el workflow `.github/workflows/urbania-refresh.yml`, que después regenera el catálogo de la web. El 20-10-2026 el sitio respondió 403 a las peticiones HTTP directas, así que hasta tener acceso cada ejecución se detiene sin datos nuevos. Para varios trabajadores, migrar a PostgreSQL y una cola. El refresco de fichas, la verificación automática de retiradas y el despliegue en servidor siguen pendientes.

Para ampliar cobertura, recorrer el catálogo de distritos sin filtros y medir IDs únicos/cobertura. No recorrer indiscriminadamente todas las combinaciones de dormitorios y precios: solapan publicaciones. Si se particiona por precios enteros, los cortes deben evitar solapamiento y mantener una consulta para atributos desconocidos. No se garantiza exhaustividad por el orden dinámico, promociones, cambios concurrentes y límites del portal.

El paquete incluye una guía de ejecución y 50 pruebas: las 43 de este notebook, 5 de la actualización periódica y 2 de la muestra. No se recapturó toda Lima ni se entrenó un modelo. El resto de “Más filtros” queda fuera de esta fase.